# PriceWise — Rolling Baseline & Pricing Pipeline

**Stage 1** clean & label → eligibility → family diagnostics → 30 pilot product-markets → daily panel  
**Stage 1.5** simple benchmarks → global weights → time-based validation → freeze Global Baseline v1  
**Stage 1.8** country / product-type / family / behavioural segmentation → compare weights → stable profiles → freeze Baseline v2  
**Stage 2** business signals → stakeholder objective weights → price adjustment → guardrails → one final recommended price

Run top to bottom. Edit `RAW_FILES` in the config cell to point at your CSV(s); all outputs go to `./outputs/`.

In [1]:
# !pip install pandas numpy scipy statsmodels holidays pyarrow
from pathlib import Path
from scipy.optimize import linprog
import holidays
import json
import numpy as np
import pandas as pd
import statsmodels.api as sm
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)

## CONFIG

```
PriceWise rolling-baseline + pricing pipeline — central configuration.

Every rule that a reviewer might want to change lives here, so the stage
scripts contain logic only. Values marked PROVISIONAL are team choices that
should be confirmed with the client (Bob) before production use.
```

In [2]:
ROOT = Path.cwd()   # outputs/ is created next to this notebook
RAW_FILES = {
    "CA": Path("/mnt/user-data/uploads/CA_valid_sales_all_transactions.csv"),
    # "US": Path(".../US_valid_sales_all_transactions.csv"),  # add when available
}
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

# ---------------------------------------------------------------- Stage 1
# Data-coverage check: a source quarter whose median weekly active-customer
# count falls below this share of the median of all other quarters is flagged
# as a coverage gap (missing data, not real demand).
COVERAGE_GAP_RATIO = 0.40

# Price lines below this unit price are treated as bundled / zero-price lines.
BUNDLED_PRICE_MAX = 1.00

NON_PRICEABLE_TYPES = {"Shipping", "Other"}       # plus missing product_type

# Product eligibility (evaluated per product-market = product_id x country).
# Two lifecycle tiers: hardware models live ~12-15 months, so a single
# "24 active months" rule would exclude every laptop. (PROVISIONAL)
HARDWARE_TYPES = {"Notebook", "Workstations", "Desktop", "Visuals", "Servers"}
ELIG_COMMON = dict(
    recent_days=90,              # must have sold in the last 90 days
    max_bundled_unit_share=0.10, # <=10% of units at bundled price
    min_price_points=10,         # distinct rounded unit prices
    price_cv_range=(0.03, 0.60), # enough variation to learn, not chaotic
)
ELIG_TIER = {
    "run_rate":  dict(min_active_months=24, first_sale_by="2024-08-31", min_units=500),
    "lifecycle": dict(min_active_months=9,  first_sale_by="2025-09-30", min_units=300),
}
N_PILOT = 30
MIN_PER_TYPE = 2                 # stratification floor per product_type

# ---------------------------------------------------------------- Stage 1.5
HORIZON = 28                     # forecast target = units in next 28 days
ORIGIN_FREQ_DAYS = 7             # a forecast origin every Monday
FIRST_ORIGIN = "2024-04-01"      # leaves 12 months of history for MA365 / LY
# Baseline components (all computed from data strictly before the origin)
COMPONENTS = ["ma7", "ma28", "ma91", "ma182", "ma365", "ly28"]
# Expanding-window validation: test blocks are calendar quarters
TEST_BLOCKS = ["2025-01", "2025-04", "2025-07", "2026-01", "2026-04", "2026-07"]
# (2025-10 block omitted: its targets fall inside the coverage gap)

# ---------------------------------------------------------------- Stage 1.8
MIN_SEGMENT_PRODUCTS = 4         # segments smaller than this fall back to global
MIN_REL_GAIN = 0.02              # segmentation must cut WAPE by >= 2% (relative)
MAX_WEIGHT_SD = 0.20             # mean across-fold SD of weights must stay below
PROFILE_MERGE_L1 = 0.35          # merge segments whose weight vectors are this close

# ---------------------------------------------------------------- Stage 2
# Stakeholder objective weights (PROVISIONAL — to be set with the client)
OBJECTIVE_WEIGHTS = dict(margin=0.40, revenue=0.25, volume=0.20, stability=0.15)
PRICE_GRID = [x / 100 for x in range(-15, 16)]   # candidate moves, -15%..+15%
ELASTICITY_BOUNDS = (-4.0, -0.2)                  # economic sanity bounds
ELASTICITY_PRIOR_SD = 0.75       # spread of true elasticities around the type-level pool
ELASTICITY_CONF_SD = 0.50        # posterior SD below this = "high" confidence

GUARDRAILS = dict(
    max_step=0.10,            # |change| <= 10% per pricing cycle
    low_conf_step=0.03,       # tighter cap when elasticity is weakly identified
    min_margin=0.05,          # price >= unit cost x 1.05
    hist_band=(0.05, 0.95),   # stay inside the trailing-365d price P5..P95 ...
    hist_slack=0.05,          # ... widened by 5%
    min_change=0.01,          # moves under 1% become "hold"
    min_recent_units=20,      # need >= 20 units in last 91 days to move price
    max_days_inactive=28,     # no sale in 28+ days = possible end-of-life -> hold & review
)
AS_OF = None                  # None = last date in the data

## BASELINE MODEL (shared by Stages 1.5, 1.8, 2)

```
Shared rolling-baseline machinery used by Stages 1.5, 1.8 and 2.

Baseline (units over the next HORIZON days) = sum_k w_k * component_k, where
each component is a simple rolling rate computed only from days strictly
before the forecast origin and scaled to the horizon:

  ma7, ma28, ma91, ma182, ma365   mean daily units over the trailing k days
  ly28                            mean daily units in the same 28-day window
                                  one year earlier (t-364 .. t-337)

Days in a coverage gap or before a product's launch are treated as unknown
(NaN), not zero. A component needs >= 50% known days in its window;
otherwise it falls back to the next-shorter component (ma365 -> ma182 ->
ma91 -> ma28; ly28 -> ma91) so that the same weights apply to every product,
including new laptops without a year of history.
```

In [3]:
WINDOWS = {"ma7": 7, "ma28": 28, "ma91": 91, "ma182": 182, "ma365": 365}
FALLBACK = {"ma365": "ma182", "ma182": "ma91", "ma91": "ma28", "ma28": "ma7", "ly28": "ma91"}

In [4]:
def prepare_series(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide matrix: rows = dates, cols = product_market; NaN for unknown days."""
    p = panel.copy()
    p["pm"] = p["product_id"] + "|" + p["country"]
    launch = p[p["units"] > 0].groupby("pm")["date"].min()
    p["prelaunch"] = p["date"] < p["pm"].map(launch)
    p.loc[p["prelaunch"], "units"] = np.nan
    return p.pivot(index="date", columns="pm", values="units").sort_index()

In [5]:
def _rolling_rate(Y: pd.DataFrame, k: int, shift: int = 1) -> pd.DataFrame:
    """Mean of Y over [t-shift-k+1 .. t-shift], i.e. strictly before t when shift=1."""
    known = Y.notna().astype(float).rolling(k, min_periods=1).sum()
    total = Y.fillna(0).rolling(k, min_periods=1).sum()
    rate = (total / known).where(known >= 0.5 * k)
    return rate.shift(shift)

In [6]:
def component_frame(Y: pd.DataFrame, origins: pd.DatetimeIndex) -> pd.DataFrame:
    """Long frame: one row per (origin, product_market) with components,
    fallbacks applied, target and validity flags."""
    comps = {name: _rolling_rate(Y, k) for name, k in WINDOWS.items()}
    comps["ly28"] = _rolling_rate(Y, 28, shift=337)       # t-364 .. t-337
    H = HORIZON
    fwd_sum = Y.fillna(0).rolling(H).sum().shift(-(H - 1))
    fwd_known = Y.notna().astype(float).rolling(H).sum().shift(-(H - 1))
    hist_days = Y.notna().astype(float).cumsum().shift(1)

    rows = []
    for name, df in comps.items():
        s = df.reindex(origins).stack(future_stack=True).rename(name)
        rows.append(s)
    out = pd.concat(rows, axis=1)
    out["target"] = fwd_sum.reindex(origins).stack(future_stack=True)
    out["target_complete"] = fwd_known.reindex(origins).stack(future_stack=True) == H
    out["history_days"] = hist_days.reindex(origins).stack(future_stack=True)
    out.index.names = ["origin", "pm"]
    out = out.reset_index()

    # Fallbacks (record which were used)
    order = ["ma7", "ma28", "ma91", "ma182", "ma365", "ly28"]
    for name in order:
        out[f"{name}_native"] = out[name].notna()
    for name in ["ma28", "ma91", "ma182", "ma365", "ly28"]:
        src = FALLBACK[name]
        out[name] = out[name].fillna(out[src])
    out[order] = out[order] * H                              # rates -> horizon units
    out["valid"] = out["target_complete"] & (out["history_days"] >= 28) & out[order].notna().all(axis=1)
    return out

In [7]:
def origins_between(start, end) -> pd.DatetimeIndex:
    return pd.date_range(start, end, freq=f"{ORIGIN_FREQ_DAYS}D")

In [8]:
def fit_weights(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Least-absolute-deviation convex combination: min sum|y - Xw|,
    w >= 0, sum w = 1. Pooled LAD == minimising WAPE on the training rows.
    Solved exactly as a linear programme."""
    n, k = X.shape
    # variables: w (k), u (n), v (n) with y - Xw = u - v, u,v >= 0
    from scipy import sparse
    c = np.r_[np.zeros(k), np.ones(n), np.ones(n)]
    top = sparse.hstack([sparse.csr_matrix(X), sparse.eye(n), -sparse.eye(n)])
    last = sparse.csr_matrix(np.r_[np.ones(k), np.zeros(2 * n)][None, :])
    A_eq = sparse.vstack([top, last]).tocsr()
    b_eq = np.r_[y, 1.0]
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=[(0, None)] * (k + 2 * n), method="highs")
    if not res.success:
        raise RuntimeError(res.message)
    w = res.x[:k]
    w[w < 1e-6] = 0
    return w / w.sum()

In [9]:
def metrics(y: np.ndarray, f: np.ndarray) -> dict:
    err = f - y
    return dict(n=len(y), MAE=np.mean(np.abs(err)), RMSE=np.sqrt(np.mean(err ** 2)),
                WAPE=np.abs(err).sum() / y.sum(), bias=err.sum() / y.sum())

In [10]:
def predict(frame: pd.DataFrame, weights: dict) -> np.ndarray:
    return frame[list(weights)].to_numpy() @ np.array(list(weights.values()))

In [11]:
def fold_masks(frame: pd.DataFrame):
    """Expanding-window, time-ordered folds. Train rows: target window fully
    observed before the test block starts. Test rows: origins inside the block."""
    for blk in TEST_BLOCKS:
        start = pd.Period(blk, "M").start_time
        end = (pd.Period(blk, "M") + 2).end_time.normalize()
        train = frame["origin"] + pd.Timedelta(days=HORIZON - 1) < start
        test = frame["origin"].between(start, end)
        yield blk, train.to_numpy(), test.to_numpy()

## STAGE 1 — data preparation

```
Stage 1 — clean & label transactions, eligibility, family diagnostics,
pilot selection, daily training panel.

Outputs (outputs/):
  s1_transactions_labeled.parquet   all rows, original values kept + label columns
  s1_coverage_by_quarter.csv        source-quarter coverage check
  s1_product_eligibility.csv        one row per product-market, every rule shown
  s1_family_diagnostics.csv         one row per product family
  s1_pilot_30.csv                   the selected pilot product-markets
  s1_daily_panel.parquet            pilot product x calendar-day panel
```

In [12]:
# ------------------------------------------------------------------ helpers

In [13]:
def sbc_class(weekly: pd.Series) -> tuple[float, float, str]:
    """Syntetos-Boylan demand class from a weekly unit series."""
    nz = weekly[weekly > 0]
    if len(nz) < 2:
        return np.nan, np.nan, "insufficient"
    adi = len(weekly) / len(nz)
    cv2 = (nz.std() / nz.mean()) ** 2
    if adi < 1.32:
        cls = "smooth" if cv2 < 0.49 else "erratic"
    else:
        cls = "intermittent" if cv2 < 0.49 else "lumpy"
    return adi, cv2, cls

In [14]:
def load(country: str) -> pd.DataFrame:
    df = pd.read_csv(RAW_FILES[country], low_memory=False)
    for c in ["order_date", "ship_date", "return_date", "week_begin"]:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    df["country"] = country
    return df

In [15]:
# ------------------------------------------------------------------ 1. clean & label

In [16]:
def clean_and_label(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    # Duplicates are labelled, not removed (per data-prep note)
    df["lbl_exact_duplicate"] = df.duplicated(keep="first")
    df["row_id"] = np.arange(len(df))
    df["unit_price"] = df["revenue"] / df["order_units"]
    df["unit_cost"] = df["cost"] / df["order_units"]

    # Labels — nothing is dropped here; downstream steps filter on labels.
    df["lbl_bundled_price"] = df["unit_price"] < BUNDLED_PRICE_MAX
    df["lbl_non_priceable_type"] = df["product_type"].isna() | df["product_type"].isin(
        NON_PRICEABLE_TYPES)
    df["lbl_negative_margin"] = df["margin"] < 0
    df["lbl_zero_cost"] = df["cost"] <= 0
    # Price outliers: outside 5x IQR of the product's own log unit price
    lp = np.log(df["unit_price"].clip(lower=1e-6))
    q1 = lp.groupby(df["product_id"]).transform(lambda s: s.quantile(0.25))
    q3 = lp.groupby(df["product_id"]).transform(lambda s: s.quantile(0.75))
    iqr = (q3 - q1).clip(lower=0.05)
    df["lbl_price_outlier"] = (~df["lbl_bundled_price"]) & ((lp < q1 - 5 * iqr) | (lp > q3 + 5 * iqr))

    # Modal family / type per product (9 products map to 2 families)
    for col in ["product_family", "product_type", "product_family_group"]:
        modal = df.groupby("product_id")[col].agg(
            lambda s: s.mode().iat[0] if s.notna().any() else np.nan)
        df[f"{col}_modal"] = df["product_id"].map(modal)

    # Coverage check by source quarter
    wk = (df.groupby(["source_file", pd.Grouper(key="order_date", freq="W-MON")])
            .agg(cust=("customer_id", "nunique"), units=("order_units", "sum"))
            .reset_index())
    q = wk.groupby("source_file").agg(median_weekly_customers=("cust", "median"),
                                      median_weekly_units=("units", "median"))
    q["ref_customers"] = [q["median_weekly_customers"].drop(i).median() for i in q.index]
    q["coverage_ratio"] = q["median_weekly_customers"] / q["ref_customers"]
    q["coverage_gap"] = q["coverage_ratio"] < COVERAGE_GAP_RATIO
    span = df.groupby("source_file")["order_date"].agg(["min", "max"])
    q = q.join(span)
    df["lbl_coverage_gap"] = df["source_file"].map(q["coverage_gap"])
    return df, q.reset_index()

In [17]:
def gap_days(cov: pd.DataFrame) -> pd.DatetimeIndex:
    days = [pd.date_range(r["min"].normalize(), r["max"].normalize())
            for _, r in cov[cov["coverage_gap"]].iterrows()]
    # quarter files span full quarters; extend to calendar quarter bounds
    out = pd.DatetimeIndex([])
    for d in days:
        out = out.union(pd.date_range(d[0].to_period("Q").start_time,
                                      d[-1].to_period("Q").end_time.normalize()))
    return out

In [18]:
# ------------------------------------------------------------------ 2. eligibility

In [19]:
def eligibility(df: pd.DataFrame, gaps: pd.DatetimeIndex) -> pd.DataFrame:
    as_of = df["order_date"].max()
    base = df[~df["lbl_coverage_gap"]]
    g = base.groupby(["country", "product_id"])
    t = g.agg(product_type=("product_type_modal", "first"),
              product_family=("product_family_modal", "first"),
              product_family_group=("product_family_group_modal", "first"),
              first_sale=("order_date", "min"), last_sale=("order_date", "max"),
              lines=("row_id", "size"), units=("order_units", "sum"),
              revenue=("revenue", "sum"), margin=("margin", "sum"),
              customers=("customer_id", "nunique"),
              active_days=("order_date", "nunique"))
    t["active_months"] = g["order_date"].apply(lambda s: s.dt.to_period("M").nunique())
    t["last_sale"] = df.groupby(["country", "product_id"])["order_date"].max()
    t["asp"] = t["revenue"] / t["units"]
    t["margin_pct"] = t["margin"] / t["revenue"]
    bund = base[base["lbl_bundled_price"]].groupby(["country", "product_id"])["order_units"].sum()
    t["bundled_unit_share"] = (bund / t["units"]).reindex(t.index).fillna(0)
    paid = base[~base["lbl_bundled_price"] & ~base["lbl_price_outlier"]]
    gp = paid.groupby(["country", "product_id"])["unit_price"]
    t["price_cv"] = gp.std() / gp.mean()
    t["price_points"] = gp.apply(lambda s: s.round(0).nunique())

    # Demand behaviour on weekly series (gap weeks removed)
    wk = (base.groupby(["country", "product_id", pd.Grouper(key="order_date", freq="W-MON")])
              ["order_units"].sum())
    beh = {}
    all_weeks = pd.date_range(base["order_date"].min(), as_of, freq="W-MON")
    all_weeks = all_weeks[~all_weeks.isin(gaps)]
    for key, s in wk.groupby(level=[0, 1]):
        s = s.droplevel([0, 1])
        s = s.reindex(all_weeks[all_weeks >= s.index.min()], fill_value=0)
        beh[key] = sbc_class(s)
    b = pd.DataFrame(beh, index=["adi", "cv2", "demand_class"]).T
    b.index.names = ["country", "product_id"]
    t = t.join(b)
    t["days_since_last_sale"] = (as_of - t["last_sale"]).dt.days

    E = ELIG_COMMON
    t["lifecycle_tier"] = np.where(t["product_type"].isin(HARDWARE_TYPES), "lifecycle", "run_rate")
    tier = lambda k: t["lifecycle_tier"].map({n: v[k] for n, v in ELIG_TIER.items()})
    rules = {
        "r_priceable_type": t["product_type"].notna() & ~t["product_type"].isin(NON_PRICEABLE_TYPES),
        "r_not_bundled": t["bundled_unit_share"] <= E["max_bundled_unit_share"],
        "r_history_length": t["first_sale"] <= pd.to_datetime(tier("first_sale_by")),
        "r_active_months": t["active_months"] >= tier("min_active_months"),
        "r_recent_sale": t["days_since_last_sale"] <= E["recent_days"],
        "r_min_units": t["units"] >= tier("min_units"),
        "r_price_points": t["price_points"] >= E["min_price_points"],
        "r_price_cv": t["price_cv"].between(*E["price_cv_range"]),
    }
    for k, v in rules.items():
        t[k] = v.fillna(False)
    rcols = list(rules)
    t["rules_passed"] = t[rcols].sum(axis=1)
    t["eligible"] = t[rcols].all(axis=1)
    t["first_failed_rule"] = np.where(
        t["eligible"], "",
        t[rcols].idxmin(axis=1).str.replace("r_", "", regex=False))
    return t.reset_index().sort_values(["eligible", "units"], ascending=False)

In [20]:
# ------------------------------------------------------------------ 3. family diagnostics

In [21]:
def family_diagnostics(df: pd.DataFrame, elig: pd.DataFrame) -> pd.DataFrame:
    base = df[~df["lbl_coverage_gap"] & ~df["lbl_non_priceable_type"]]
    fam = base.groupby(["country", "product_family_modal"])
    d = fam.agg(product_types=("product_type_modal", lambda s: ", ".join(sorted(s.dropna().unique()))),
                products=("product_id", "nunique"), units=("order_units", "sum"),
                revenue=("revenue", "sum"), margin=("margin", "sum"),
                customers=("customer_id", "nunique"))
    d.index.names = ["country", "product_family"]
    d["margin_pct"] = d["margin"] / d["revenue"]
    d["bundled_line_share"] = fam["lbl_bundled_price"].mean().values
    d["negative_margin_line_share"] = fam["lbl_negative_margin"].mean().values

    # Seasonality: November share of units relative to a flat 1/12
    m = base.assign(mon=base["order_date"].dt.month)
    nov = m[m["mon"] == 11].groupby(["country", "product_family_modal"])["order_units"].sum()
    d["november_index"] = (nov.values / d["units"].values) * 12 if len(nov) == len(d) else \
        (nov.reindex(d.index, fill_value=0) / d["units"] * 12).values
    # YoY growth: Apr-Aug 2026 vs Apr-Aug 2025 (both outside the gap)
    def window(a, b):
        w = base[(base["order_date"] >= a) & (base["order_date"] <= b)]
        return w.groupby(["country", "product_family_modal"])["order_units"].sum()
    cur, prev = window("2026-04-01", "2026-08-28"), window("2025-04-01", "2025-08-28")
    cur.index.names = prev.index.names = ["country", "product_family"]
    d["yoy_units_growth"] = (cur / prev - 1).reindex(d.index)

    e = elig.groupby(["country", "product_family"])
    d["median_price_cv"] = e["price_cv"].median()
    d["eligible_products"] = e["eligible"].sum()
    d["share_products_eligible"] = d["eligible_products"] / d["products"]
    cls = elig[elig["eligible"]].pivot_table(index=["country", "product_family"],
                                             columns="demand_class", values="product_id",
                                             aggfunc="count", fill_value=0)
    cls.columns = [f"eligible_{c}" for c in cls.columns]
    d = d.join(cls).fillna({c: 0 for c in cls.columns})
    d["pilot_suitability"] = np.select(
        [d["eligible_products"] >= 5, d["eligible_products"] >= 1],
        ["good", "thin"], "not suitable")
    return d.reset_index().sort_values("revenue", ascending=False)

In [22]:
# ------------------------------------------------------------------ 4. pilot selection

In [23]:
def select_pilot(elig: pd.DataFrame) -> pd.DataFrame:
    """Stratified by product_type: floor of MIN_PER_TYPE per type (where
    eligible), remaining slots by revenue-proportional quota, filled in order
    of units within type."""
    e = elig[elig["eligible"]].copy()
    picks = []
    for country, ec in e.groupby("country"):
        ec = ec.sort_values("units", ascending=False)
        rev = ec.groupby("product_type")["revenue"].sum()
        avail = ec.groupby("product_type").size()
        quota = np.minimum(avail, MIN_PER_TYPE)
        left = N_PILOT - quota.sum()
        share = rev / rev.sum()
        while left > 0:
            gap = (share * N_PILOT - quota).where(quota < avail, -np.inf)
            k = gap.idxmax()
            quota[k] += 1
            left -= 1
        for ptype, n in quota.items():
            picks.append(ec[ec["product_type"] == ptype].head(int(n)))
    p = pd.concat(picks)
    p["pilot_rank"] = p.groupby("country")["units"].rank(ascending=False, method="first").astype(int)
    p["product_market"] = p["product_id"] + "|" + p["country"]
    return p.sort_values(["country", "pilot_rank"])

In [24]:
# ------------------------------------------------------------------ 5. daily panel

In [25]:
def daily_panel(df: pd.DataFrame, pilot: pd.DataFrame, gaps: pd.DatetimeIndex) -> pd.DataFrame:
    keys = pilot[["country", "product_id"]]
    tx = df.merge(keys, on=["country", "product_id"])
    paid = tx[~tx["lbl_bundled_price"]]
    day = (tx.groupby(["country", "product_id", "order_date"])
             .agg(units=("order_units", "sum"), revenue=("revenue", "sum"),
                  cost=("cost", "sum"), lines=("row_id", "size"),
                  customers=("customer_id", "nunique")))
    pp = paid.groupby(["country", "product_id", "order_date"]).agg(
        paid_units=("order_units", "sum"), paid_revenue=("revenue", "sum"))
    day = day.join(pp)
    dates = pd.date_range(df["order_date"].min(), df["order_date"].max(), freq="D")
    idx = pd.MultiIndex.from_frame(keys).repeat(len(dates))
    full = pd.DataFrame(index=idx).reset_index()
    full["date"] = np.tile(dates, len(keys))
    day = day.reset_index().rename(columns={"order_date": "date"})
    p = full.merge(day, on=["country", "product_id", "date"], how="left")
    num = ["units", "revenue", "cost", "lines", "customers", "paid_units", "paid_revenue"]
    p[num] = p[num].fillna(0)

    p["coverage_gap"] = p["date"].isin(gaps)
    # Mask gap days: demand unknown, not zero
    p.loc[p["coverage_gap"], ["units", "revenue", "cost", "paid_units", "paid_revenue"]] = np.nan

    p["asp"] = p["paid_revenue"] / p["paid_units"].replace(0, np.nan)
    p["unit_cost"] = p["cost"] / p["units"].replace(0, np.nan)
    p = p.sort_values(["country", "product_id", "date"])
    g = p.groupby(["country", "product_id"])
    # Price known at the START of the day = last observed ASP before today
    p["last_price"] = g["asp"].transform(lambda s: s.ffill().shift(1))
    p["last_unit_cost"] = g["unit_cost"].transform(lambda s: s.ffill().shift(1))
    sold = p["units"].fillna(0) > 0
    last_sale = p["date"].where(sold).groupby([p["country"], p["product_id"]]).ffill()
    p["days_since_last_sale"] = (p["date"] - last_sale.groupby([p["country"], p["product_id"]]).shift(1)).dt.days

    ca_h = holidays.country_holidays("CA", years=range(2023, 2027))
    p["dow"] = p["date"].dt.dayofweek
    p["month"] = p["date"].dt.month
    fq = p["date"].dt.to_period("Q-MAR")          # Lenovo FY starts in April
    p["fiscal_quarter"] = "FY" + fq.dt.qyear.astype(str).str[-2:] + "Q" + fq.dt.quarter.astype(str)
    p["is_holiday"] = p["date"].dt.date.map(lambda d: d in ca_h)
    p["is_month_end"] = p["date"].dt.is_month_end
    p["is_quarter_end"] = p["date"].dt.is_quarter_end
    p["is_black_friday_week"] = False
    for y in range(2023, 2027):
        nov = pd.date_range(f"{y}-11-01", f"{y}-11-30")
        bf = nov[(nov.dayofweek == 4)][3]
        p.loc[p["date"].between(bf - pd.Timedelta(days=4), bf + pd.Timedelta(days=3)),
              "is_black_friday_week"] = True
    attrs = pilot.set_index(["country", "product_id"])[
        ["product_type", "product_family", "product_family_group", "demand_class", "lifecycle_tier"]]
    p = p.join(attrs, on=["country", "product_id"])
    return p

In [26]:
def stage1_main():
    frames, covs = [], []
    for ctry in RAW_FILES:
        df = load(ctry)
        df, cov = clean_and_label(df)
        cov.insert(0, "country", ctry)
        frames.append(df)
        covs.append(cov)
    df = pd.concat(frames, ignore_index=True)
    cov = pd.concat(covs, ignore_index=True)
    gaps = gap_days(cov)

    df.to_parquet(OUT / "s1_transactions_labeled.parquet", index=False)
    cov.to_csv(OUT / "s1_coverage_by_quarter.csv", index=False)

    elig = eligibility(df, gaps)
    elig.to_csv(OUT / "s1_product_eligibility.csv", index=False)
    fam = family_diagnostics(df, elig)
    fam.to_csv(OUT / "s1_family_diagnostics.csv", index=False)
    pilot = select_pilot(elig)
    pilot.to_csv(OUT / "s1_pilot_30.csv", index=False)
    panel = daily_panel(df, pilot, gaps)
    panel.to_parquet(OUT / "s1_daily_panel.parquet", index=False)
    pd.Series(gaps.strftime("%Y-%m-%d")).to_csv(OUT / "s1_gap_days.csv", index=False, header=["date"])

    lbl = [c for c in df.columns if c.startswith("lbl_")]
    print("rows", len(df)); print(df[lbl].sum())
    print(cov[["source_file", "median_weekly_customers", "coverage_ratio", "coverage_gap"]])
    print("eligible:", elig["eligible"].sum(), "of", len(elig))
    print(elig["first_failed_rule"].value_counts())
    print(fam.drop(columns=["country"]).round(3).to_string())
    print(pilot[["pilot_rank", "product_id", "product_type", "product_family", "units",
                 "active_months", "price_cv", "demand_class", "lifecycle_tier", "revenue"]].to_string())
    print("panel", panel.shape)

### Run Stage 1

In [27]:
stage1_main()
pilot = pd.read_csv(OUT / 's1_pilot_30.csv')
display(pd.read_csv(OUT / 's1_coverage_by_quarter.csv')[['source_file','median_weekly_customers','coverage_ratio','coverage_gap']])
display(pd.read_csv(OUT / 's1_family_diagnostics.csv').round(3))
display(pilot[['pilot_rank','product_id','product_type','product_family','lifecycle_tier','demand_class','units','active_months','price_cv']])

rows 498263
lbl_exact_duplicate       13909
lbl_bundled_price         39826
lbl_non_priceable_type      111
lbl_negative_margin       89671
lbl_zero_cost               852
lbl_price_outlier          7138
lbl_coverage_gap          16572
dtype: int64
               source_file  median_weekly_customers  coverage_ratio  coverage_gap
0   synthesized_FY24Q1.csv                      8.0        0.410256         False
1   synthesized_FY24Q2.csv                     11.0        0.564103         False
2   synthesized_FY24Q3.csv                     10.5        0.538462         False
3   synthesized_FY24Q4.csv                     14.0        0.717949         False
4   synthesized_FY25Q1.csv                     15.5        0.794872         False
5   synthesized_FY25Q2.csv                     15.5        0.794872         False
6   synthesized_FY25Q3.csv                     19.5        1.258065         False
7   synthesized_FY25Q4.csv                     23.0        1.483871         False
8   synthesiz

,source_file,median_weekly_customers,coverage_ratio,coverage_gap
0,synthesized_FY24Q1.csv,8.0,0.410256,False
1,synthesized_FY24Q2.csv,11.0,0.564103,False
2,synthesized_FY24Q3.csv,10.5,0.538462,False
3,synthesized_FY24Q4.csv,14.0,0.717949,False
4,synthesized_FY25Q1.csv,15.5,0.794872,False
5,synthesized_FY25Q2.csv,15.5,0.794872,False
6,synthesized_FY25Q3.csv,19.5,1.258065,False
7,synthesized_FY25Q4.csv,23.0,1.483871,False
8,synthesized_FY26Q1.csv,21.0,1.354839,False
9,synthesized_FY26Q2.csv,25.0,1.612903,False


,country,product_family,product_types,products,units,revenue,margin,customers,margin_pct,bundled_line_share,negative_margin_line_share,november_index,yoy_units_growth,median_price_cv,eligible_products,share_products_eligible,eligible_erratic,eligible_intermittent,eligible_lumpy,eligible_smooth,pilot_suitability
0,CA,Product Family Frame Road,Notebook,531,80464.0,1.071009e+08,2.619075e+07,210,0.245,0.000,0.142,1.364,-0.388,0.106,16,0.030,7.0,0.0,4.0,5.0,good
1,CA,Product Family Solar Peak,"Notebook, Workstations",630,55040.0,8.999577e+07,1.714596e+07,202,0.191,0.000,0.178,1.405,-0.501,0.107,5,0.008,3.0,0.0,2.0,0.0,good
2,CA,Product Family Knight Thrive,Notebook,447,77794.0,6.256619e+07,9.750983e+06,180,0.156,0.000,0.195,1.159,0.207,0.087,15,0.034,3.0,1.0,10.0,1.0,good
3,CA,Product Family Romeo Iris,"Options, Services",1383,390961.0,2.767504e+07,1.284744e+07,389,0.464,0.142,0.195,1.080,-0.157,0.124,73,0.053,46.0,1.0,10.0,16.0,good
4,CA,Product Family Titan Onset,Desktop,303,25826.0,2.120949e+07,3.952818e+06,128,0.186,0.000,0.195,0.740,-0.381,0.128,3,0.010,2.0,0.0,1.0,0.0,thin
5,CA,Product Family Quest Lunar,Visuals,137,46277.0,9.038410e+06,1.997301e+06,155,0.221,0.000,0.106,1.231,0.136,0.154,6,0.044,2.0,0.0,4.0,0.0,good
6,CA,Product Family Circuit Insight,Servers,73,640.0,1.935609e+06,8.117202e+05,14,0.419,0.000,0.033,0.994,-0.934,0.262,0,0.000,0.0,0.0,0.0,0.0,not suitable
7,CA,Product Family Harbor Momentum,"Notebook, Servers, Services",146,652.0,4.054417e+05,1.690850e+05,11,0.417,0.000,0.072,1.288,-0.731,0.114,0,0.000,0.0,0.0,0.0,0.0,not suitable
8,CA,Product Family Signal Road,Options,22,1136.0,9.243798e+04,-3.366207e+04,11,-0.364,0.005,0.880,3.285,-0.833,0.154,0,0.000,0.0,0.0,0.0,0.0,not suitable
9,CA,Product Family Stream Juliet,Desktop,4,31.0,7.570756e+04,4.077764e+04,6,0.539,0.000,0.077,0.000,6.000,0.013,0,0.000,0.0,0.0,0.0,0.0,not suitable


,pilot_rank,product_id,product_type,product_family,lifecycle_tier,demand_class,units,active_months,price_cv
0,1,27WH3132DN,Options,Product Family Romeo Iris,run_rate,smooth,17366.0,36,0.183408
1,2,9TE3M17113,Services,Product Family Romeo Iris,run_rate,smooth,10036.0,38,0.549576
2,3,4Q01B61187,Options,Product Family Romeo Iris,run_rate,erratic,5444.0,34,0.326395
3,4,2B70T80159,Options,Product Family Romeo Iris,run_rate,erratic,5441.0,38,0.248244
4,5,55C97203QD,Options,Product Family Romeo Iris,run_rate,smooth,4177.0,36,0.200802
5,6,5YO5U33136,Services,Product Family Romeo Iris,run_rate,smooth,3963.0,38,0.186588
6,7,5IQ5M98979,Services,Product Family Romeo Iris,run_rate,erratic,3157.0,34,0.150663
7,8,13BASGW8QU,Visuals,Product Family Quest Lunar,lifecycle,erratic,1358.0,32,0.235213
8,9,19Y55740YG,Notebook,Product Family Knight Thrive,lifecycle,lumpy,1331.0,24,0.117339
9,10,01Y5AUP2ZB,Visuals,Product Family Quest Lunar,lifecycle,lumpy,1278.0,21,0.197966


## STAGE 1.5 — Global Baseline v1

```
Stage 1.5 — simple benchmarks, global weights, time-based validation,
freeze Global Baseline v1.

Outputs (outputs/):
  s15_component_frame.parquet  origin x product rows with components + target
  s15_validation_by_fold.csv   every model x fold
  s15_validation_summary.csv   pooled across folds, overall and by tier
  s15_weights_by_fold.csv      global weights learned in each fold (stability)
  baseline_v1.json             frozen Global Baseline v1
```

In [28]:
def build_frame():
    panel = pd.read_parquet(OUT / "s1_daily_panel.parquet")
    Y = prepare_series(panel)
    last = Y.index.max() - pd.Timedelta(days=HORIZON - 1)
    origins = origins_between(FIRST_ORIGIN, last)
    fr = component_frame(Y, origins)
    pilot = pd.read_csv(OUT / "s1_pilot_30.csv")
    attrs = pilot.set_index("product_market")[["product_type", "product_family", "demand_class",
                                               "lifecycle_tier", "country"]]
    fr = fr.join(attrs, on="pm")
    return fr, Y

In [29]:
def stage15_main():
    fr, Y = build_frame()
    fr.to_parquet(OUT / "s15_component_frame.parquet", index=False)
    v = fr[fr["valid"]].reset_index(drop=True)
    print("valid rows", len(v), "of", len(fr), "| products", v["pm"].nunique(),
          "| origins", v["origin"].nunique())

    comps = COMPONENTS
    benchmarks = {f"bench_{c}": {c: 1.0} for c in comps}
    benchmarks["bench_equal_weight"] = {c: 1 / len(comps) for c in comps}

    rows, wrows, preds = [], [], []
    for blk, tr, te in fold_masks(v):
        if te.sum() == 0:
            continue
        w = fit_weights(v.loc[tr, comps].to_numpy(), v.loc[tr, "target"].to_numpy())
        wd = dict(zip(comps, w))
        wrows.append({"fold": blk, "train_rows": int(tr.sum()), **wd})
        models = {**benchmarks, "global_v1": wd}
        test = v[te]
        for name, wts in models.items():
            f = predict(test, wts)
            rows.append({"fold": blk, "model": name, **metrics(test["target"].to_numpy(), f)})
            preds.append(test[["origin", "pm", "lifecycle_tier", "product_type", "target"]]
                         .assign(model=name, forecast=f, fold=blk))

    byfold = pd.DataFrame(rows)
    P = pd.concat(preds)
    P.to_parquet(OUT / "s15_test_predictions.parquet", index=False)

    def summarise(df, keys):
        return (df.groupby(keys).apply(lambda g: pd.Series(metrics(g["target"].to_numpy(),
                                                                    g["forecast"].to_numpy())),
                                       include_groups=False).reset_index())
    summ = summarise(P.assign(segment="ALL"), ["segment", "model"])
    summ_t = summarise(P.rename(columns={"lifecycle_tier": "segment"}), ["segment", "model"])
    summary = pd.concat([summ, summ_t]).sort_values(["segment", "WAPE"])
    W = pd.DataFrame(wrows)

    byfold.to_csv(OUT / "s15_validation_by_fold.csv", index=False)
    summary.to_csv(OUT / "s15_validation_summary.csv", index=False)
    W.to_csv(OUT / "s15_weights_by_fold.csv", index=False)

    # Freeze v1: refit on every valid row available
    w_final = dict(zip(comps, fit_weights(v[comps].to_numpy(), v["target"].to_numpy())))
    g = summary[(summary.segment == "ALL")].set_index("model")
    best_bench = g.drop(index="global_v1")["WAPE"].idxmin()
    frozen = {
        "version": "global_baseline_v1",
        "frozen_on": str(pd.Timestamp.today().date()),
        "data_through": str(Y.index.max().date()),
        "horizon_days": HORIZON,
        "components": comps,
        "fallbacks": FALLBACK,
        "weights": {k: round(float(x), 4) for k, x in w_final.items()},
        "fit": "pooled least-absolute-deviation, convex weights (w>=0, sum=1)",
        "validation": {
            "scheme": "expanding window, quarterly test blocks " + ", ".join(TEST_BLOCKS),
            "test_rows": int(g.loc["global_v1", "n"]),
            "WAPE": round(float(g.loc["global_v1", "WAPE"]), 4),
            "MAE": round(float(g.loc["global_v1", "MAE"]), 2),
            "bias": round(float(g.loc["global_v1", "bias"]), 4),
            "best_simple_benchmark": best_bench,
            "best_simple_benchmark_WAPE": round(float(g.loc[best_bench, "WAPE"]), 4),
        },
        "weight_stability_mean_sd_across_folds": round(float(W[comps].std().mean()), 4),
    }
    (OUT / "baseline_v1.json").write_text(json.dumps(frozen, indent=2))

    pd.set_option("display.width", 200)
    print(summary.round(3).to_string(index=False))
    print(W.round(3).to_string(index=False))
    print(json.dumps(frozen, indent=2))

### Run Stage 1.5

In [30]:
stage15_main()
display(pd.read_csv(OUT / 's15_validation_summary.csv').round(3))
display(pd.read_csv(OUT / 's15_weights_by_fold.csv').round(3))
print((OUT / 'baseline_v1.json').read_text())

valid rows 1901 of 3660 | products 30 | origins 105


  segment              model      n     MAE    RMSE  WAPE   bias
      ALL          global_v1 1462.0  44.939  74.421 0.453  0.016
      ALL         bench_ma28 1462.0  45.513  76.921 0.459 -0.010
      ALL bench_equal_weight 1462.0  45.793  75.441 0.462  0.004
      ALL          bench_ma7 1462.0  48.243  81.872 0.486 -0.012
      ALL         bench_ma91 1462.0  50.072  86.217 0.505  0.045
      ALL        bench_ma182 1462.0  52.881  90.811 0.533  0.064
      ALL        bench_ma365 1462.0  55.512  89.047 0.560  0.007
      ALL         bench_ly28 1462.0  64.452 102.089 0.650 -0.071
lifecycle          global_v1 1007.0  35.591  53.851 0.619 -0.042
lifecycle bench_equal_weight 1007.0  35.940  53.851 0.625 -0.048
lifecycle         bench_ma28 1007.0  37.292  58.261 0.648 -0.039
lifecycle          bench_ma7 1007.0  38.786  63.498 0.674 -0.035
lifecycle         bench_ma91 1007.0  39.547  58.642 0.687 -0.030
lifecycle        bench_ma182 1007.0  40.571  59.050 0.705 -0.038
lifecycle        bench_ma

,segment,model,n,MAE,RMSE,WAPE,bias
0,ALL,global_v1,1462.0,44.939,74.421,0.453,0.016
1,ALL,bench_ma28,1462.0,45.513,76.921,0.459,-0.010
2,ALL,bench_equal_weight,1462.0,45.793,75.441,0.462,0.004
3,ALL,bench_ma7,1462.0,48.243,81.872,0.486,-0.012
4,ALL,bench_ma91,1462.0,50.072,86.217,0.505,0.045
5,ALL,bench_ma182,1462.0,52.881,90.811,0.533,0.064
6,ALL,bench_ma365,1462.0,55.512,89.047,0.560,0.007
7,ALL,bench_ly28,1462.0,64.452,102.089,0.650,-0.071
8,lifecycle,global_v1,1007.0,35.591,53.851,0.619,-0.042
9,lifecycle,bench_equal_weight,1007.0,35.940,53.851,0.625,-0.048


,fold,train_rows,ma7,ma28,ma91,ma182,ma365,ly28
0,2025-01,383,0.000,0.530,0.470,0.000,0.000,0.000
1,2025-04,565,0.176,0.000,0.710,0.000,0.000,0.114
2,2025-07,747,0.171,0.069,0.611,0.031,0.000,0.118
3,2026-01,1001,0.202,0.222,0.325,0.130,0.000,0.120
4,2026-04,1271,0.234,0.229,0.281,0.147,0.000,0.108
5,2026-07,1661,0.288,0.147,0.399,0.047,0.044,0.075


{
  "version": "global_baseline_v1",
  "frozen_on": "2026-10-22",
  "data_through": "2026-08-28",
  "horizon_days": 28,
  "components": [
    "ma7",
    "ma28",
    "ma91",
    "ma182",
    "ma365",
    "ly28"
  ],
  "fallbacks": {
    "ma365": "ma182",
    "ma182": "ma91",
    "ma91": "ma28",
    "ma28": "ma7",
    "ly28": "ma91"
  },
  "weights": {
    "ma7": 0.3355,
    "ma28": 0.2741,
    "ma91": 0.27,
    "ma182": 0.0,
    "ma365": 0.0494,
    "ly28": 0.071
  },
  "fit": "pooled least-absolute-deviation, convex weights (w>=0, sum=1)",
  "validation": {
    "scheme": "expanding window, quarterly test blocks 2025-01, 2025-04, 2025-07, 2026-01, 2026-04, 2026-07",
    "test_rows": 1462,
    "WAPE": 0.453,
    "MAE": 44.94,
    "bias": 0.0161,
    "best_simple_benchmark": "bench_ma28",
    "best_simple_benchmark_WAPE": 0.4588
  },
  "weight_stability_mean_sd_across_folds": 0.0964
}


## STAGE 1.8 — segmentation & Baseline v2

```
Stage 1.8 — test segmentation schemes, compare segmented weights, select a
small number of stable profiles, freeze Baseline v2.

Schemes tested: country, product_type, product_family, lifecycle_tier
(behavioural: hardware lifecycle vs run-rate) and demand_class (behavioural:
Syntetos-Boylan smooth/erratic/intermittent/lumpy).

A segment with fewer than MIN_SEGMENT_PRODUCTS products uses global weights.
A scheme is accepted only if it (a) cuts pooled validation WAPE by at least
MIN_REL_GAIN vs Global v1 and (b) keeps weights stable across folds
(mean SD <= MAX_WEIGHT_SD). Accepted segments with similar weight vectors are
merged into profiles, then the profile model is re-validated before freezing.

Outputs: s18_scheme_comparison.csv, s18_segment_weights.csv,
         s18_profile_validation.csv, baseline_v2.json
```

In [31]:
SCHEMES = ["country", "product_type", "product_family", "lifecycle_tier", "demand_class"]

In [32]:
def fit_segmented(v, seg_col, tr):
    """Weights per segment from training rows; small segments -> global."""
    comps = COMPONENTS
    glob = fit_weights(v.loc[tr, comps].to_numpy(), v.loc[tr, "target"].to_numpy())
    out = {}
    for s, g in v[tr].groupby(seg_col):
        if g["pm"].nunique() >= MIN_SEGMENT_PRODUCTS:
            out[s] = fit_weights(g[comps].to_numpy(), g["target"].to_numpy())
    return glob, out

In [33]:
def predict_segmented(test, seg_col, glob, segw):
    comps = COMPONENTS
    W = np.vstack([segw.get(s, glob) for s in test[seg_col]])
    return (test[comps].to_numpy() * W).sum(axis=1)

In [34]:
def evaluate_scheme(v, seg_col):
    preds, wrows = [], []
    for blk, tr, te in fold_masks(v):
        if te.sum() == 0:
            continue
        glob, segw = fit_segmented(v, seg_col, tr)
        f = predict_segmented(v[te], seg_col, glob, segw)
        preds.append(v.loc[te, ["pm", "origin", "target", seg_col]].assign(forecast=f, fold=blk))
        for s, w in segw.items():
            wrows.append({"fold": blk, "segment": s, **dict(zip(COMPONENTS, w))})
    P = pd.concat(preds)
    W = pd.DataFrame(wrows)
    m = metrics(P["target"].to_numpy(), P["forecast"].to_numpy())
    sd = W.groupby("segment")[COMPONENTS].std().mean(axis=1) if len(W) else pd.Series(dtype=float)
    n_segs = v[seg_col].nunique()
    n_own = len(sd)
    return m, sd, W, P, n_segs, n_own

In [35]:
def merge_profiles(final_w: dict) -> dict:
    """Greedy agglomerative merge of segments whose weight vectors are within
    PROFILE_MERGE_L1 (L1 distance). Returns segment -> profile name."""
    groups = [[s] for s in final_w]
    cent = {tuple(g): final_w[g[0]] for g in groups}
    merged = True
    while merged and len(groups) > 1:
        merged = False
        best = None
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                d = np.abs(cent[tuple(groups[i])] - cent[tuple(groups[j])]).sum()
                if d < PROFILE_MERGE_L1 and (best is None or d < best[0]):
                    best = (d, i, j)
        if best:
            _, i, j = best
            g = groups[i] + groups[j]
            cent[tuple(g)] = (cent[tuple(groups[i])] * len(groups[i]) +
                              cent[tuple(groups[j])] * len(groups[j])) / len(g)
            groups = [x for k, x in enumerate(groups) if k not in (i, j)] + [g]
            merged = True
    return {s: f"P{k + 1}" for k, g in enumerate(groups) for s in g}

In [36]:
def stage18_main():
    v = pd.read_parquet(OUT / "s15_component_frame.parquet")
    v = v[v["valid"]].reset_index(drop=True)
    v1 = json.loads((OUT / "baseline_v1.json").read_text())
    v1_wape = v1["validation"]["WAPE"]

    rows, seg_rows = [], []
    for sc in SCHEMES:
        m, sd, W, P, n_segs, n_own = evaluate_scheme(v, sc)
        gain = (v1_wape - m["WAPE"]) / v1_wape
        stab = float(sd.mean()) if len(sd) else np.nan
        testable = n_own >= 2
        rows.append({"scheme": sc, "segments": n_segs, "segments_with_own_weights": n_own,
                     "WAPE": m["WAPE"], "MAE": m["MAE"], "bias": m["bias"],
                     "rel_gain_vs_v1": gain, "mean_weight_sd": stab,
                     "accepted": bool(testable and gain >= MIN_REL_GAIN and stab <= MAX_WEIGHT_SD),
                     "note": "" if testable else "fewer than 2 segments large enough; equals global"})
        if len(W):
            last = W[W["fold"] == W["fold"].max()].assign(scheme=sc)
            last = last.join(sd.rename("weight_sd"), on="segment")
            seg_rows.append(last)
    comp = pd.DataFrame(rows).sort_values("WAPE")
    segw = pd.concat(seg_rows)
    comp.to_csv(OUT / "s18_scheme_comparison.csv", index=False)
    segw.to_csv(OUT / "s18_segment_weights.csv", index=False)
    pd.set_option("display.width", 220)
    print(comp.round(4).to_string(index=False))
    print(segw.round(3).to_string(index=False))

    acc = comp[comp["accepted"]]
    if acc.empty:
        chosen, profile_map = None, {}
        print("No scheme accepted; v2 = v1 weights.")
    else:
        chosen = acc.iloc[0]["scheme"]
        comps = COMPONENTS
        # final per-segment weights on all data -> merge into profiles
        glob, sw = fit_segmented(v, chosen, np.ones(len(v), bool))
        profile_map = merge_profiles(sw)
    # Products whose segment had no own weights map to the global profile
    v["profile"] = v[chosen].map(profile_map).fillna("P0_global") if chosen else "P0_global"

    # Re-validate the profile model with the same folds
    m, sd, W, P, *_ = evaluate_scheme(v, "profile")
    prof_val = pd.DataFrame([{"model": "global_v1", "WAPE": v1_wape},
                             {"model": "profiles_v2", **m}])
    # per-tier comparison
    Pv1 = pd.read_parquet(OUT / "s15_test_predictions.parquet")
    Pv1 = Pv1[Pv1["model"].isin(["global_v1", "bench_ma28"])]
    P = P.join(v.drop_duplicates("pm").set_index("pm")["lifecycle_tier"], on="pm")
    tiers = []
    for t, g in P.groupby("lifecycle_tier"):
        tiers.append({"model": "profiles_v2", "segment": t, **metrics(g.target.values, g.forecast.values)})
    for (mod, t), g in Pv1.groupby(["model", "lifecycle_tier"]):
        tiers.append({"model": mod, "segment": t, **metrics(g.target.values, g.forecast.values)})
    prof_val = pd.concat([prof_val.assign(segment="ALL"), pd.DataFrame(tiers)])
    prof_val.to_csv(OUT / "s18_profile_validation.csv", index=False)
    P.to_parquet(OUT / "s18_test_predictions.parquet", index=False)
    print(prof_val.round(4).to_string(index=False))

    # Freeze v2 (fit profile weights on all valid rows)
    adopt = chosen is not None and m["WAPE"] < v1_wape
    profiles = {}
    if adopt:
        for prof, g in v.groupby("profile"):
            if g["pm"].nunique() >= MIN_SEGMENT_PRODUCTS:
                w = fit_weights(g[COMPONENTS].to_numpy(), g["target"].to_numpy())
            else:
                w = np.array(list(v1["weights"].values()))
            members = sorted(k for k, p in profile_map.items() if p == prof)
            profiles[prof] = {"assign_when": {chosen: members} if members else "otherwise",
                              "weights": {k: round(float(x), 4) for k, x in zip(COMPONENTS, w)},
                              "pilot_products": int(g["pm"].nunique())}
    else:
        profiles["P0_global"] = {"assign_when": "all", "weights": v1["weights"],
                                 "pilot_products": int(v["pm"].nunique())}
    profiles.setdefault("P0_global", {"assign_when": "fallback for any product not matched",
                                      "weights": v1["weights"]})
    assign = v.drop_duplicates("pm").set_index("pm")["profile"] if adopt else \
        pd.Series("P0_global", index=v["pm"].unique())
    frozen = {
        "version": "baseline_v2",
        "frozen_on": str(pd.Timestamp.today().date()),
        "data_through": v1["data_through"],
        "segmentation_scheme": chosen if adopt else None,
        "schemes_tested": SCHEMES,
        "acceptance_rule": f"rel. WAPE gain >= {MIN_REL_GAIN:.0%} vs v1 and mean weight SD <= {MAX_WEIGHT_SD}",
        "profiles": profiles,
        "pilot_assignment": assign.to_dict(),
        "validation": {"WAPE": round(float(m["WAPE"]), 4), "MAE": round(float(m["MAE"]), 2),
                       "bias": round(float(m["bias"]), 4), "v1_WAPE": v1_wape},
    }
    (OUT / "baseline_v2.json").write_text(json.dumps(frozen, indent=2))
    print(json.dumps({k: frozen[k] for k in ["segmentation_scheme", "profiles", "validation"]}, indent=2))

### Run Stage 1.8

In [37]:
stage18_main()
display(pd.read_csv(OUT / 's18_scheme_comparison.csv').round(4))
display(pd.read_csv(OUT / 's18_segment_weights.csv').round(3))
print((OUT / 'baseline_v2.json').read_text()[:1500])

        scheme  segments  segments_with_own_weights   WAPE     MAE   bias  rel_gain_vs_v1  mean_weight_sd  accepted                                              note
lifecycle_tier         2                          2 0.4524 44.8843 0.0174          0.0012          0.0855     False                                                  
       country         1                          1 0.4530 44.9389 0.0161          0.0000          0.0964     False fewer than 2 segments large enough; equals global
  demand_class         3                          3 0.4546 45.0976 0.0206         -0.0035          0.0936     False                                                  
product_family         6                          3 0.4555 45.1827 0.0129         -0.0054          0.1071     False                                                  
  product_type         6                          2 0.4565 45.2848 0.0179         -0.0077          0.1581     False                                                  
   f

      model   WAPE      n     MAE     RMSE    bias   segment
  global_v1 0.4530    NaN     NaN      NaN     NaN       ALL
profiles_v2 0.4530 1462.0 44.9389  74.4212  0.0161       ALL
profiles_v2 0.6186 1007.0 35.5911  53.8511 -0.0424 lifecycle
profiles_v2 0.3429  455.0 65.6273 106.6686  0.0550  run_rate
 bench_ma28 0.6481 1007.0 37.2923  58.2609 -0.0390 lifecycle
 bench_ma28 0.3328  455.0 63.7057 107.2358  0.0096  run_rate
  global_v1 0.6186 1007.0 35.5911  53.8511 -0.0424 lifecycle
  global_v1 0.3429  455.0 65.6273 106.6686  0.0550  run_rate
{
  "segmentation_scheme": null,
  "profiles": {
    "P0_global": {
      "assign_when": "all",
      "weights": {
        "ma7": 0.3355,
        "ma28": 0.2741,
        "ma91": 0.27,
        "ma182": 0.0,
        "ma365": 0.0494,
        "ly28": 0.071
      },
      "pilot_products": 30
    }
  },
  "validation": {
    "WAPE": 0.453,
    "MAE": 44.94,
    "bias": 0.0161,
    "v1_WAPE": 0.453
  }
}


,scheme,segments,segments_with_own_weights,WAPE,MAE,bias,rel_gain_vs_v1,mean_weight_sd,accepted,note
0,lifecycle_tier,2,2,0.4524,44.8843,0.0174,0.0012,0.0855,False,NaN
1,country,1,1,0.4530,44.9389,0.0161,0.0000,0.0964,False,fewer than 2 segments large enough; equals global
2,demand_class,3,3,0.4546,45.0976,0.0206,-0.0035,0.0936,False,NaN
3,product_family,6,3,0.4555,45.1827,0.0129,-0.0054,0.1071,False,NaN
4,product_type,6,2,0.4565,45.2848,0.0179,-0.0077,0.1581,False,NaN


,fold,segment,ma7,ma28,ma91,ma182,ma365,ly28,scheme,weight_sd
0,2026-07,CA,0.288,0.147,0.399,0.047,0.044,0.075,country,0.096
1,2026-07,Notebook,0.350,0.039,0.000,0.547,0.063,0.000,product_type,0.214
2,2026-07,Options,0.303,0.085,0.463,0.000,0.080,0.069,product_type,0.102
3,2026-07,Product Family Frame Road,0.246,0.000,0.000,0.000,0.238,0.515,product_family,0.143
4,2026-07,Product Family Knight Thrive,0.424,0.053,0.094,0.428,0.000,0.000,product_family,0.087
5,2026-07,Product Family Romeo Iris,0.234,0.162,0.355,0.121,0.000,0.128,product_family,0.091
6,2026-07,lifecycle,0.392,0.200,0.222,0.144,0.041,0.000,lifecycle_tier,0.080
7,2026-07,run_rate,0.234,0.162,0.355,0.121,0.000,0.128,lifecycle_tier,0.091
8,2026-07,erratic,0.384,0.228,0.000,0.000,0.273,0.116,demand_class,0.127
9,2026-07,lumpy,0.344,0.155,0.162,0.338,0.000,0.000,demand_class,0.065


{
  "version": "baseline_v2",
  "frozen_on": "2026-10-22",
  "data_through": "2026-08-28",
  "segmentation_scheme": null,
  "schemes_tested": [
    "country",
    "product_type",
    "product_family",
    "lifecycle_tier",
    "demand_class"
  ],
  "acceptance_rule": "rel. WAPE gain >= 2% vs v1 and mean weight SD <= 0.2",
  "profiles": {
    "P0_global": {
      "assign_when": "all",
      "weights": {
        "ma7": 0.3355,
        "ma28": 0.2741,
        "ma91": 0.27,
        "ma182": 0.0,
        "ma365": 0.0494,
        "ly28": 0.071
      },
      "pilot_products": 30
    }
  },
  "pilot_assignment": {
    "01Y5AUP2ZB|CA": "P0_global",
    "13BASGW8QU|CA": "P0_global",
    "27WH3132DN|CA": "P0_global",
    "2B70T80159|CA": "P0_global",
    "4Q01B61187|CA": "P0_global",
    "55C97203QD|CA": "P0_global",
    "5IQ5M98979|CA": "P0_global",
    "5YO5U33136|CA": "P0_global",
    "9TE3M17113|CA": "P0_global",
    "19Y55740YG|CA": "P0_global",
    "87JI6103KJ|CA": "P0_global",
    "06AS87

## STAGE 2 — final recommended price

```
Stage 2 — business signals -> stakeholder objective weights -> price
adjustment -> guardrails -> one final recommended price per pilot product.

1. Signals (as of the last data date): current price, unit cost, margin,
   Baseline v2 demand for the next 28 days, momentum, YoY, price position,
   price elasticity (product estimate shrunk toward its product-type pool),
   customer concentration.
2. Objectives: for every candidate price p0*(1+x), x in PRICE_GRID, expected
   units q = baseline * (1+x)^elasticity. Score each objective as % change vs
   holding price: margin $, revenue $, units, and stability (-|x|).
3. Adjustment: x* = argmax sum_i weight_i * score_i (stakeholder weights).
4. Guardrails (in order): data sufficiency, step cap (tighter when elasticity
   confidence is low), margin floor, trailing-365d price band, minimum change,
   rounding. Each binding guardrail is recorded.

Outputs: s2_business_signals.csv, s2_objective_grid.csv,
         s2_price_recommendations.csv
```

In [38]:
# ------------------------------------------------------------------ signals

In [39]:
def elasticities(panel: pd.DataFrame) -> pd.DataFrame:
    p = panel[~panel["coverage_gap"]].copy()
    p["wk"] = p["date"].dt.to_period("W-SUN").dt.start_time
    w = (p.groupby(["product_id", "country", "product_type", "wk"])
           .agg(u=("paid_units", "sum"), r=("paid_revenue", "sum")).reset_index())
    w = w[w["u"] > 0].copy()
    w["lp"], w["lq"] = np.log(w["r"] / w["u"]), np.log(w["u"])
    w["t"] = (w["wk"] - w.groupby("product_id")["wk"].transform("min")).dt.days / 365
    w["moy"] = w["wk"].dt.month

    rows = []
    for (pid, ctry, ptype), g in w.groupby(["product_id", "country", "product_type"]):
        X = sm.add_constant(g[["lp", "t"]])
        r = sm.OLS(g["lq"], X).fit(cov_type="HC1")
        rows.append(dict(product_id=pid, country=ctry, product_type=ptype, weeks=len(g),
                         e_raw=r.params["lp"], e_raw_se=r.bse["lp"]))
    e = pd.DataFrame(rows)
    # Pooled within-product estimate per product type (product-demeaned)
    pool = {}
    for ptype, g in w.groupby("product_type"):
        d = g[["lq", "lp", "t"]] - g.groupby("product_id")[["lq", "lp", "t"]].transform("mean")
        r = sm.OLS(d["lq"], d[["lp", "t"]]).fit(cov_type="cluster",
                                                 cov_kwds={"groups": g["product_id"].astype("category").cat.codes})
        pool[ptype] = (r.params["lp"], r.bse["lp"])
    e["e_pool"] = e["product_type"].map(lambda k: pool[k][0])
    e["e_pool_se"] = e["product_type"].map(lambda k: pool[k][1])
    # Precision-weighted shrinkage toward the pool
    prior_var = ELASTICITY_PRIOR_SD ** 2 + e["e_pool_se"] ** 2
    prec_i, prec_0 = 1 / e["e_raw_se"] ** 2, 1 / prior_var
    e["e_post"] = (e["e_raw"] * prec_i + e["e_pool"] * prec_0) / (prec_i + prec_0)
    e["e_post_sd"] = np.sqrt(1 / (prec_i + prec_0))
    e["shrink_weight_on_product"] = prec_i / (prec_i + prec_0)
    e["elasticity"] = e["e_post"].clip(*ELASTICITY_BOUNDS)
    e["elasticity_clipped"] = e["elasticity"] != e["e_post"]
    e["elasticity_confidence"] = np.where(e["e_post_sd"] < ELASTICITY_CONF_SD, "high", "low")
    return e

In [40]:
def business_signals(tx: pd.DataFrame, panel: pd.DataFrame, pilot: pd.DataFrame,
                     v2: dict, as_of: pd.Timestamp) -> pd.DataFrame:
    keys = pilot[["product_id", "country"]]
    t = tx.merge(keys, on=["product_id", "country"])
    t = t[~t["lbl_bundled_price"] & ~t["lbl_price_outlier"] & ~t["lbl_coverage_gap"]]

    def win(days):
        return t[(t["order_date"] > as_of - pd.Timedelta(days=days)) & (t["order_date"] <= as_of)]

    k = ["product_id", "country"]
    out = pilot.set_index(k)[["product_market", "product_type", "product_family",
                              "lifecycle_tier", "demand_class"]].copy()
    for d in (28, 91, 365):
        g = win(d).groupby(k)
        out[f"units_{d}d"] = g["order_units"].sum()
        out[f"asp_{d}d"] = g["revenue"].sum() / g["order_units"].sum()
    out[["units_28d", "units_91d", "units_365d"]] = out[["units_28d", "units_91d", "units_365d"]].fillna(0)
    out["current_price"] = out["asp_28d"].where(out["units_28d"] >= 5, out["asp_91d"])
    out["current_price_basis"] = np.where(out["units_28d"] >= 5, "28d ASP", "91d ASP")
    g91 = win(91).groupby(k)
    out["unit_cost"] = g91["cost"].sum() / g91["order_units"].sum()
    out["margin_pct"] = 1 - out["unit_cost"] / out["current_price"]
    g365 = win(365).groupby(k)["unit_price"]
    out["price_p5_365d"] = g365.quantile(GUARDRAILS["hist_band"][0])
    out["price_p50_365d"] = g365.median()
    out["price_p95_365d"] = g365.quantile(GUARDRAILS["hist_band"][1])
    out["price_index_vs_365d_median"] = out["current_price"] / out["price_p50_365d"]
    # customer concentration (top customer share of 91d units)
    cu = win(91).groupby(k + ["customer_id"])["order_units"].sum()
    out["top_customer_share_91d"] = cu.groupby(level=[0, 1]).max() / cu.groupby(level=[0, 1]).sum()
    out["active_customers_91d"] = cu.groupby(level=[0, 1]).size()
    # YoY on 91d units
    ly = t[(t["order_date"] > as_of - pd.Timedelta(days=91 + 364)) &
           (t["order_date"] <= as_of - pd.Timedelta(days=364))].groupby(k)["order_units"].sum()
    out["yoy_units_91d"] = out["units_91d"] / ly - 1
    out["days_since_last_sale"] = (as_of - t.groupby(k)["order_date"].max()).dt.days

    # Baseline v2 for the next 28 days, origin = day after as_of
    Y = prepare_series(panel)
    origin = as_of + pd.Timedelta(days=1)
    Y = Y.reindex(pd.date_range(Y.index.min(), origin))
    fr = component_frame(Y, pd.DatetimeIndex([origin])).set_index("pm")
    prof_of = v2["pilot_assignment"]
    prof = v2["profiles"]
    bl = {}
    for pm, r in fr.iterrows():
        w = prof[prof_of.get(pm, "P0_global")]["weights"]
        bl[pm] = sum(r[c] * wt for c, wt in w.items())
    out["baseline_units_28d"] = out["product_market"].map(bl)
    out["baseline_profile"] = out["product_market"].map(lambda x: prof_of.get(x, "P0_global"))
    out["momentum_28_vs_91"] = out["product_market"].map(fr["ma28"] / fr["ma91"]) - 1
    return out.reset_index()

In [41]:
# ------------------------------------------------------------------ objectives

In [42]:
def objective_grid(sig: pd.DataFrame) -> pd.DataFrame:
    W = OBJECTIVE_WEIGHTS
    rows = []
    for _, s in sig.iterrows():
        p0, c, e, q0 = s["current_price"], s["unit_cost"], s["elasticity"], s["baseline_units_28d"]
        R0, M0 = p0 * q0, (p0 - c) * q0
        for x in PRICE_GRID:
            p = p0 * (1 + x)
            q = q0 * (1 + x) ** e
            R, M = p * q, (p - c) * q
            sc = dict(margin=(M - M0) / abs(M0) if M0 > 0 else (M - M0) / R0,
                      revenue=R / R0 - 1, volume=q / q0 - 1, stability=-abs(x))
            rows.append(dict(product_market=s["product_market"], move=x, price=p, units=q,
                             revenue=R, margin=M, **{f"score_{k}": v for k, v in sc.items()},
                             utility=sum(W[k] * sc[k] for k in W)))
    return pd.DataFrame(rows)

In [43]:
# ------------------------------------------------------------------ guardrails

In [44]:
def round_price(p, p0):
    """Whole dollars at/above $100, dimes below; always rounds toward the
    current price so rounding can never breach a guardrail."""
    unit = 1.0 if p >= 100 else 0.1
    f = np.floor if p > p0 else np.ceil
    return round(float(f(p / unit - 1e-9 if p > p0 else p / unit + 1e-9) * unit), 2)

In [45]:
def apply_guardrails(sig: pd.DataFrame, grid: pd.DataFrame) -> pd.DataFrame:
    G = GUARDRAILS
    best = grid.loc[grid.groupby("product_market")["utility"].idxmax()].set_index("product_market")
    recs = []
    for _, s in sig.iterrows():
        pm, p0 = s["product_market"], s["current_price"]
        x_raw = best.loc[pm, "move"]
        x, binds = x_raw, []
        if s["days_since_last_sale"] >= G["max_days_inactive"]:
            x, binds = 0.0, [f"no sale in {int(s['days_since_last_sale'])} days - possible end-of-life, review"]
        elif s["units_91d"] < G["min_recent_units"] or pd.isna(s["baseline_units_28d"]):
            x, binds = 0.0, ["insufficient recent data"]
        else:
            cap = G["max_step"] if s["elasticity_confidence"] == "high" else G["low_conf_step"]
            if abs(x) > cap:
                x = np.sign(x) * cap
                binds.append(f"step cap ±{cap:.0%}" + (" (low elasticity confidence)"
                                                       if cap == G["low_conf_step"] else ""))
            p = p0 * (1 + x)
            lo = s["price_p5_365d"] * (1 - G["hist_slack"])
            hi = s["price_p95_365d"] * (1 + G["hist_slack"])
            if p < lo or p > hi:
                p = min(max(p, lo), hi)
                binds.append("trailing-365d price band")
            floor = s["unit_cost"] * (1 + G["min_margin"])
            if p < floor:
                p = min(floor, p0 * (1 + G["max_step"]))
                binds.append(f"margin floor (cost +{G['min_margin']:.0%})")
            x = p / p0 - 1
            if abs(x) < G["min_change"]:
                if x != 0:
                    binds.append(f"change < {G['min_change']:.0%} -> hold")
                x = 0.0
        final = p0 if x == 0 else round_price(p0 * (1 + x), p0)
        xf = final / p0 - 1
        q0 = s["baseline_units_28d"]
        q1 = q0 * (1 + xf) ** s["elasticity"] if pd.notna(q0) else np.nan
        c = s["unit_cost"]
        action = "hold" if xf == 0 else ("increase" if xf > 0 else "decrease")
        recs.append(dict(
            product_market=pm, product_id=s["product_id"], country=s["country"],
            product_type=s["product_type"], product_family=s["product_family"],
            current_price=round(p0, 2), unit_cost=round(c, 2),
            unconstrained_move=x_raw, recommended_price=round(final, 2),
            change_pct=xf, action=action,
            guardrails_binding="; ".join(binds) if binds else "none",
            elasticity=s["elasticity"], elasticity_confidence=s["elasticity_confidence"],
            baseline_units_28d=q0, expected_units_28d=q1,
            expected_revenue_change_28d=final * q1 - p0 * q0,
            expected_margin_change_28d=(final - c) * q1 - (p0 - c) * q0,
            margin_pct_before=1 - c / p0, margin_pct_after=1 - c / final,
        ))
    r = pd.DataFrame(recs)
    r["rationale"] = r.apply(rationale, axis=1)
    return r

In [46]:
def rationale(r):
    e = r["elasticity"]
    sens = "price-sensitive" if e < -1.5 else ("moderately price-sensitive" if e < -1 else "fairly price-insensitive")
    if r["action"] == "hold":
        why = "objectives roughly balance at the current price" if r["guardrails_binding"] == "none" \
            else f"held by guardrail: {r['guardrails_binding']}"
        return f"Hold. Demand is {sens} (elasticity {e:.2f}); {why}."
    direction = "Raise" if r["action"] == "increase" else "Lower"
    return (f"{direction} {abs(r['change_pct']):.1%}. Demand is {sens} (elasticity {e:.2f}, "
            f"{r['elasticity_confidence']} confidence); expected 28-day margin "
            f"{r['expected_margin_change_28d']:+,.0f} and revenue {r['expected_revenue_change_28d']:+,.0f}. "
            f"Guardrails: {r['guardrails_binding']}.")

In [47]:
def stage2_main():
    tx = pd.read_parquet(OUT / "s1_transactions_labeled.parquet",
                         columns=["product_id", "country", "order_date", "customer_id", "order_units",
                                  "revenue", "cost", "unit_price", "lbl_bundled_price",
                                  "lbl_price_outlier", "lbl_coverage_gap"])
    panel = pd.read_parquet(OUT / "s1_daily_panel.parquet")
    pilot = pd.read_csv(OUT / "s1_pilot_30.csv")
    v2 = json.loads((OUT / "baseline_v2.json").read_text())
    as_of = pd.Timestamp(AS_OF) if AS_OF else tx["order_date"].max()

    sig = business_signals(tx, panel, pilot, v2, as_of)
    el = elasticities(panel)
    sig = sig.merge(el.drop(columns=["product_type"]), on=["product_id", "country"], how="left")
    sig.insert(0, "as_of", as_of.date())
    sig.to_csv(OUT / "s2_business_signals.csv", index=False)

    grid = objective_grid(sig)
    grid.to_csv(OUT / "s2_objective_grid.csv", index=False)
    rec = apply_guardrails(sig, grid)
    rec.insert(0, "as_of", as_of.date())
    rec.insert(1, "objective_weights", json.dumps(OBJECTIVE_WEIGHTS))
    rec.to_csv(OUT / "s2_price_recommendations.csv", index=False)

    pd.set_option("display.width", 250)
    print(sig[["product_id", "product_type", "current_price", "unit_cost", "margin_pct", "baseline_units_28d",
               "units_28d", "momentum_28_vs_91", "e_raw", "e_raw_se", "e_pool", "elasticity",
               "elasticity_confidence"]].round(3).to_string(index=False))
    print(rec[["product_id", "product_type", "current_price", "recommended_price", "change_pct",
               "unconstrained_move", "guardrails_binding", "expected_margin_change_28d",
               "expected_revenue_change_28d"]].round(3).to_string(index=False))
    print(rec["action"].value_counts())
    print("total exp margin chg", rec.expected_margin_change_28d.sum().round(0),
          "rev chg", rec.expected_revenue_change_28d.sum().round(0))

### Run Stage 2

In [48]:
stage2_main()
rec = pd.read_csv(OUT / 's2_price_recommendations.csv')
display(rec[['product_id','product_type','current_price','recommended_price','change_pct','action','elasticity','elasticity_confidence','guardrails_binding','expected_margin_change_28d','expected_revenue_change_28d']].round(3))

product_id product_type  current_price  unit_cost  margin_pct  baseline_units_28d  units_28d  momentum_28_vs_91  e_raw  e_raw_se  e_pool  elasticity elasticity_confidence
27WH3132DN      Options        144.143    108.420       0.248              55.735        0.0             -1.000 -1.957     0.711  -2.455      -2.177                   low
9TE3M17113     Services        148.973      3.231       0.978             120.518       89.0             -0.346 -0.607     0.303  -1.396      -0.690                  high
4Q01B61187      Options         19.430      9.649       0.503             383.410      338.0             -0.000 -2.243     0.464  -2.455      -2.296                  high
2B70T80159      Options         18.047     11.486       0.364             102.642      123.0              1.332 -1.868     0.338  -2.455      -1.957                  high
55C97203QD      Options        322.398    178.090       0.448              15.775        0.0             -1.000 -2.774     0.530  -2.455      -2.

,product_id,product_type,current_price,recommended_price,change_pct,action,elasticity,elasticity_confidence,guardrails_binding,expected_margin_change_28d,expected_revenue_change_28d
0,27WH3132DN,Options,144.14,144.14,0.000,hold,-2.177,low,"no sale in 88 days - possible end-of-life, review",0.000,0.000
1,9TE3M17113,Services,148.97,148.97,0.000,hold,-0.690,high,none,0.000,0.000
2,4Q01B61187,Options,19.43,17.50,-0.099,decrease,-2.296,high,step cap ±10%,77.382,1081.951
3,2B70T80159,Options,18.05,16.30,-0.097,decrease,-1.957,high,step cap ±10%,-70.336,189.670
4,55C97203QD,Options,322.40,322.40,0.000,hold,-2.676,high,"no sale in 66 days - possible end-of-life, review",0.000,0.000
5,5YO5U33136,Services,96.40,93.60,-0.029,decrease,-1.863,low,step cap ±3% (low elasticity confidence),-37.632,345.648
6,5IQ5M98979,Services,156.46,156.46,0.000,hold,-2.011,low,"no sale in 79 days - possible end-of-life, review",0.000,0.000
7,13BASGW8QU,Visuals,217.21,217.21,0.000,hold,-2.202,low,"no sale in 38 days - possible end-of-life, review",0.000,0.000
8,19Y55740YG,Notebook,878.62,904.00,0.029,increase,-2.309,low,step cap ±3% (low elasticity confidence),1544.032,-2787.489
9,01Y5AUP2ZB,Visuals,226.45,226.45,0.000,hold,-1.829,low,"no sale in 74 days - possible end-of-life, review",0.000,0.000
